# RQ3.07 — PC-shift representativeness & concentrated misclassifications

Two analyses supporting the RQ3.02 latent-alignment narrative.

**Part 1 — why latent alignment helps Chip 02 and hurts Chip 01.** PC recentering
applies a single per-chip offset `Δ_c = ref_pc_embed − own_pc_embed` to *every*
embedding on that chip. That only helps if the PC well's offset is representative
of how each *target's* own centroid has moved. This part measures both the
displacement magnitude `‖Δ_c‖` per chip and the cosine similarity between `Δ_c`
and each target's own centroid displacement `δ_(c,t)`.

**Part 2 — do concentrated misclassifications point at the nearest training
target?** A programmatic remake of the manual `tab:target_similarity_summary`
table, but keyed on confusions **>50%** of a true class's test pixels (rather
than the eyeballed "dominant" cells) for `cnn_gru_dual_attn_recon`. For each such
confusion it asks whether the predicted class is the *nearest* training-pool
target by curve shape, by TTP, or by either.

Read-only with respect to the pipeline — imports only. Note the distance helpers
in Part 2 are copied from `RQ3_04_loco_result_analysis.ipynb` (they live in
notebook cells, not an importable module); if they drift, re-sync from there.

In [ ]:
import os, sys, importlib, math, gc, re
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist
from sklearn.metrics import confusion_matrix

try:
    notebook_path = globals().get('__vsc_ipynb_file__')
    if notebook_path:
        os.chdir(os.path.dirname(os.path.dirname(notebook_path)))
except Exception as e:
    print(f"Could not change directory: {e}")

print("CWD:", os.getcwd())
%load_ext autoreload
%autoreload 2
import config

sys.path.insert(0, 'utils')
sys.path.insert(0, 'utils/model_training')
from model_utils import build_neighbor_curve_stack, _QuerySlice, _AttnScores, _WeightedRecon

cdt   = importlib.import_module("04_cross_dataset_training")
p08   = importlib.import_module("08_cross_dataset_predict_new_chip")
vis07 = importlib.import_module("07_attribution_vis_all")
rio   = importlib.import_module("cross_dataset_result_io")
b06   = importlib.import_module("06b_cross_dataset_prediction_report")

import tensorflow as tf
tf.config.optimizer.set_jit(False)

%matplotlib inline
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.linestyle': '--', 'grid.alpha': 0.3, 'font.size': 9})
print("Imports OK.")

In [ ]:
MODEL             = "cnn_gru_dual_attn_recon"
GROUP_NAME        = "final_6_new"
TRAIN_CENTER_FRAC = 0.5
CURVE_TYPE        = "ori_curve_sg_p4_norm"
CURVE_ALIGNMENT   = "pc_ttp"
PC_TTP_ANCHOR     = "min"
EXP_FOLDER        = config.FINAL_EXP_FOLDER
OUTLIER_FILTER    = "noamp_remove"
MODE_STR          = "lofo"
BATCH_N           = 800     # pixels sampled per chip for the embedding centroids
EXCLUDE_LABELS    = ("PC", "NC-ALL")

folder_names = config.CROSS_DATASET_GROUPS[GROUP_NAME]
exp_paths = [Path(EXP_FOLDER, name) for name in folder_names]

def short_name(folder):
    m = re.search(r'DDM_0(\d)', folder)
    return f'Chip 0{m.group(1)}' if m else folder.split('_U_', 1)[1]

def chip_sort_key(folder_or_label):
    m = re.search(r'DDM_0(\d)', folder_or_label)
    return int(m.group(1)) if m else folder_or_label

CHIPS = sorted(folder_names, key=chip_sort_key)

def group_dir(group_name):
    return Path(EXP_FOLDER) / "cross_dataset_cv" / group_name

def lofo_group_dir(group_name):
    return b06.alignment_dir(group_dir(group_name), CURVE_ALIGNMENT, PC_TTP_ANCHOR)

pc_ttp_cache_dir = lofo_group_dir(GROUP_NAME) / "_cache_pc_ttp"

def load_lofo_results(group_name, curve_type, train_center_frac=TRAIN_CENTER_FRAC):
    out_dir = lofo_group_dir(group_name)
    legacy_path = b06.find_results_path(out_dir, MODE_STR, curve_type)
    return b06.load_partitioned(out_dir, MODE_STR, curve_type, legacy_path=legacy_path,
                                train_center_frac=train_center_frac)

lofo_results = load_lofo_results(GROUP_NAME, CURVE_TYPE)
print(f"{len(CHIPS)} chips, {sum(1 for c in CHIPS if f'lofo_{c}' in lofo_results)} folds with results")

## Part 1 — is the PC offset representative of the target offsets?

`pc_recenter` adds one vector `Δ_c = ref_pc_embed − own_pc_embed` to every embedding
on chip *c* ([08_cross_dataset_predict_new_chip.py:222-236](08_cross_dataset_predict_new_chip.py#L222-L236)).
It can only help if that PC-derived offset points the same way as the offsets the
actual targets need.

For each held-out chip we compute, in the fold model's embedding space:

- `Δ_c` — the recentering shift actually applied, and its magnitude `‖Δ_c‖`
- `δ_(c,t)` — for each target *t*, `train_centroid_t − chip_centroid_t`, i.e. the
  displacement that target would need to be corrected by
- `cos(Δ_c, δ_(c,t))` — how well the PC offset stands in for that target's offset

A high mean cosine means the single PC offset is a good proxy for all targets
(alignment should help); a low or negative one means it points the wrong way.

In [ ]:
def mapped_labels(chip, Y_well_raw):
    mapping = config.LABEL_MAPPINGS[chip]
    return np.array([mapping.get(w, w) for w in Y_well_raw], dtype=object)


def fold_embeddings(held_out, rng):
    """Embeddings for every chip under the fold model trained with `held_out` held out.

    Mirrors RQ3_05's load_fold + model_embeddings, and 08's pc_recenter, so the shift
    computed here is the one actually applied at inference."""
    out_dir = lofo_group_dir(GROUP_NAME)
    model_dir = out_dir / "model_interpretation" / f"lofo_{held_out}"

    resampler_path = p08._resolve_alignment_path(
        out_dir, config.CROSS_DATASET_RESAMPLER_PATH, CURVE_TYPE, held_out)
    if not resampler_path.exists():
        print(f"  [!] {short_name(held_out)}: no resampler, skipping."); return None
    resampler = joblib.load(resampler_path)

    models = vis07.load_saved_models(model_dir, OUTLIER_FILTER, len(resampler.t_grid),
                                     curve_type=CURVE_TYPE, model_names=[MODEL],
                                     train_center_frac=TRAIN_CENTER_FRAC)
    model = models.get(MODEL) if isinstance(models, dict) else None
    if model is None:
        print(f"  [!] {short_name(held_out)}: {MODEL} not saved for this fold, skipping."); return None

    k = p08._infer_k(model)
    per_chip = {}
    for chip in CHIPS:
        res = p08.align_new_chip(Path(EXP_FOLDER, chip), out_dir, CURVE_TYPE, CURVE_ALIGNMENT,
                                 PC_TTP_ANCHOR, group_name=GROUP_NAME, held_out_chip=held_out)
        if res is None:
            continue
        curves, _, Y_well_raw, pc_curves, coords, well_ids = res
        if coords is None or well_ids is None:
            continue
        n = min(BATCH_N, len(curves))
        idx = rng.choice(len(curves), size=n, replace=False)
        stack = build_neighbor_curve_stack(curves[idx], coords[idx], well_ids[idx], k)
        emb = p08.compute_embeddings_stack(model, stack)
        own_pc = (p08.compute_embeddings_stack(model, p08._pc_mean_stack(pc_curves, k))[0]
                  if pc_curves is not None and len(pc_curves) else None)
        per_chip[chip] = dict(emb=emb, labels=mapped_labels(chip, Y_well_raw)[idx], own_pc=own_pc)

    ref_embed = p08.reference_pc_embedding(model, MODEL, exp_paths, out_dir, CURVE_TYPE,
                                           OUTLIER_FILTER, CURVE_ALIGNMENT, held_out_chip=held_out)
    tf.keras.backend.clear_session()
    return per_chip, ref_embed


def _cos(a, b):
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    return float(np.dot(a, b) / (na * nb)) if na > 0 and nb > 0 else np.nan


rng = np.random.default_rng(42)
shift_rows, cos_rows = [], []

for held_out in CHIPS:
    print(f"[fold] {short_name(held_out)}")
    got = fold_embeddings(held_out, rng)
    if got is None:
        continue
    per_chip, ref_embed = got
    d = per_chip.get(held_out)
    if d is None or d["own_pc"] is None:
        print(f"  [!] {short_name(held_out)}: no own PC embedding, skipping."); continue

    delta_c = ref_embed - d["own_pc"]                      # the shift pc_recenter applies
    shift_rows.append({"chip": short_name(held_out), "shift_norm": float(np.linalg.norm(delta_c))})

    # training-pool centroids per target (every chip except the held-out one)
    train_emb, train_lab = [], []
    for c, dd in per_chip.items():
        if c == held_out:
            continue
        train_emb.append(dd["emb"]); train_lab.append(dd["labels"])
    train_emb = np.concatenate(train_emb); train_lab = np.concatenate(train_lab)

    for t in sorted(set(d["labels"]) - set(EXCLUDE_LABELS)):
        m_chip = d["labels"] == t
        m_train = train_lab == t
        if m_chip.sum() == 0 or m_train.sum() == 0:
            continue
        delta_t = train_emb[m_train].mean(axis=0) - d["emb"][m_chip].mean(axis=0)
        cos_rows.append({"chip": short_name(held_out), "target": t,
                         "cos": _cos(delta_c, delta_t),
                         "target_shift_norm": float(np.linalg.norm(delta_t))})
    gc.collect()

shift_df = pd.DataFrame(shift_rows).set_index("chip")
cos_df = pd.DataFrame(cos_rows)
print("\ndone")

In [ ]:
print("PC recentering displacement magnitude per chip:\n")
print(shift_df.round(3).to_string())

piv = cos_df.pivot_table(index="chip", columns="target", values="cos")
piv["mean"] = piv.mean(axis=1)
print("\n\ncos(Delta_c, delta_(c,t)) -- PC offset vs each target's own centroid displacement:\n")
print(piv.round(3).to_string())

print("\n\nValues for the RQ3.02 sentence:")
for chip in ("Chip 02", "Chip 01"):
    if chip in piv.index:
        print(f"  {chip}: mean cosine = {piv.loc[chip, 'mean']:+.3f}   "
              f"(per target: " + ", ".join(f"{t}={piv.loc[chip, t]:+.2f}"
                                            for t in piv.columns if t != "mean"
                                            and not np.isnan(piv.loc[chip, t])) + ")")

fig, ax = plt.subplots(figsize=(9, 4.5))
order = sorted(piv.index, key=chip_sort_key)
x = np.arange(len(order))
targets = [c for c in piv.columns if c != "mean"]
w = 0.8 / len(targets)
for i, t in enumerate(targets):
    ax.bar(x + (i - (len(targets)-1)/2)*w, [piv.loc[c, t] for c in order], width=w, label=t)
ax.plot(x, [piv.loc[c, "mean"] for c in order], "k--o", lw=1.6, ms=5, label="mean", zorder=5)
ax.axhline(0, color="black", lw=1)
ax.set_xticks(x); ax.set_xticklabels(order)
ax.set_ylabel("cos($\\Delta_c$, $\\delta_{c,t}$)", fontsize=11)
ax.set_title("Is the PC offset representative of each target's offset?",
             fontsize=12, fontweight="bold")
ax.legend(fontsize=8, ncol=3); ax.grid(alpha=0.3, axis="y")
plt.tight_layout(); plt.show()

## Part 2 — concentrated misclassifications vs. the nearest training target

Programmatic remake of `tab:target_similarity_summary`. Two differences from the
manual version:

1. **Selection is objective**: every off-diagonal confusion where the model sends
   **more than 50%** of a true class's held-out pixels to one wrong class,
   rather than eyeballed "dominant" cells.
2. **Model is fixed** to `cnn_gru_dual_attn_recon`.

For each such confusion `true → pred`, we ask whether `pred` is the **nearest**
training-pool target to `true`:

- **Curve shape** — mean euclidean distance between per-well mean curves
- **TTP** — mean euclidean distance between per-well mean `Ct`
- **Either** — nearest under at least one of the two

Distance helpers below are copied from `RQ3_04_loco_result_analysis.ipynb`
(cells 9/11/15), where they live inside notebook cells rather than a module.

In [ ]:
def build_pool(held_out_chip, curve_type=CURVE_TYPE):
    return cdt.combine_group_pc_aligned(
        exp_paths, GROUP_NAME, curve_type, held_out_chip=held_out_chip,
        anchor_method=PC_TTP_ANCHOR, anchor_pct=cdt.PC_TTP_ANCHOR_PCT_DEFAULT,
        pc_ttp_cache_dir=pc_ttp_cache_dir)


def build_well_table(combined):
    df = pd.DataFrame({"well_id": combined["well_ids"], "dataset_id": combined["dataset_id"],
                       "label": combined["Y_mapped"]})
    df["row_idx"] = np.arange(len(df))
    meta = df.groupby("well_id").agg(dataset_id=("dataset_id", "first"), label=("label", "first"))
    row_idx_map = df.groupby("well_id")["row_idx"].apply(np.array)
    wells = meta.join(row_idx_map.rename("row_idx")).reset_index()
    return wells


def compute_label_pair_distance_matrix(combined, held_out_chip, metric='euclidean'):
    """Held-out target x training target, mean distance between per-well mean curves."""
    wells = build_well_table(combined)
    wells = wells[~wells["label"].isin(EXCLUDE_LABELS)].reset_index(drop=True)
    well_curves = np.stack([combined["curves"][idx].mean(axis=0) for idx in wells["row_idx"]])
    well_labels = wells["label"].to_numpy()
    is_held = wells["dataset_id"].to_numpy() == held_out_chip

    test_c, test_l = well_curves[is_held], well_labels[is_held]
    train_c, train_l = well_curves[~is_held], well_labels[~is_held]
    row_labels, col_labels = sorted(set(test_l)), sorted(set(train_l))
    mean_mat = np.full((len(row_labels), len(col_labels)), np.nan)
    for i, li in enumerate(row_labels):
        Xi = test_c[test_l == li]
        for j, lj in enumerate(col_labels):
            Xj = train_c[train_l == lj]
            if len(Xi) and len(Xj):
                mean_mat[i, j] = cdist(Xi, Xj, metric=metric).ravel().mean()
    return row_labels, col_labels, mean_mat


def load_ct_pool(exp_paths, curve_type, group_name):
    parts = {k: [] for k in ("Ct", "dataset_id", "well_ids", "Y_mapped")}
    for exp_path in exp_paths:
        data_path = os.path.join(exp_path, config.TRAINING_DATA_PATH)
        if not os.path.exists(data_path):
            continue
        data = config.apply_well_exclusion(joblib.load(data_path), exp_path.name, group_name=group_name)
        try:
            idx, _ = config.resolve_curve_dataset_idx(curve_type, list(data["dataset_name"]))
        except ValueError:
            continue
        if "Ct" not in data["kinetic_features"][idx].columns:
            continue
        mapping = config.get_label_mappings(exp_path).get(exp_path.name)
        if mapping is None:
            continue
        Y_well_raw = np.asarray(data["Y_well"])
        ct = data["kinetic_features"][idx]["Ct"].to_numpy()
        parts["Ct"].append(ct)
        parts["dataset_id"].append(np.full(len(ct), exp_path.name, dtype=object))
        parts["well_ids"].append(np.array([f"{exp_path.name}::{w}" for w in Y_well_raw], dtype=object))
        parts["Y_mapped"].append(np.array([mapping.get(w, w) for w in Y_well_raw], dtype=object))
    if not parts["Ct"]:
        return None
    return {k: np.concatenate(v, axis=0) for k, v in parts.items()}


def compute_label_pair_ct_matrix(ct_pool, held_out_chip):
    df = pd.DataFrame({"well_id": ct_pool["well_ids"], "dataset_id": ct_pool["dataset_id"],
                       "label": ct_pool["Y_mapped"], "Ct": ct_pool["Ct"]})
    df = df[~df["label"].isin(EXCLUDE_LABELS)]
    well_ct = df.groupby("well_id").agg(dataset_id=("dataset_id", "first"),
                                        label=("label", "first"), Ct=("Ct", "mean")).reset_index()
    is_held = well_ct["dataset_id"].to_numpy() == held_out_chip
    ct, lab = well_ct["Ct"].to_numpy(), well_ct["label"].to_numpy()
    test_ct, test_l = ct[is_held], lab[is_held]
    train_ct, train_l = ct[~is_held], lab[~is_held]
    row_labels, col_labels = sorted(set(test_l)), sorted(set(train_l))
    mean_mat = np.full((len(row_labels), len(col_labels)), np.nan)
    for i, li in enumerate(row_labels):
        Xi = test_ct[test_l == li].reshape(-1, 1)
        for j, lj in enumerate(col_labels):
            Xj = train_ct[train_l == lj].reshape(-1, 1)
            if len(Xi) and len(Xj):
                mean_mat[i, j] = cdist(Xi, Xj, metric='euclidean').ravel().mean()
    return row_labels, col_labels, mean_mat


curve_dist, ct_dist = {}, {}
ct_pool = load_ct_pool(exp_paths, CURVE_TYPE, GROUP_NAME)
for held_out in CHIPS:
    combined = build_pool(held_out)
    if combined is not None:
        curve_dist[held_out] = compute_label_pair_distance_matrix(combined, held_out)
        del combined; gc.collect()
    if ct_pool is not None:
        ct_dist[held_out] = compute_label_pair_ct_matrix(ct_pool, held_out)
    print(f"  distances done: {short_name(held_out)}")

In [ ]:
MIN_CONFUSION = 0.50   # >50% of a true class's held-out pixels sent to one wrong class


def confusion_matrix_for(held_out, model=MODEL, filter_name=OUTLIER_FILTER):
    fold = lofo_results.get(f"lofo_{held_out}", {})
    res = fold.get(filter_name)
    class_names = fold.get("class_names")
    preds_key = config.MODEL_KEY_MAP.get(model, (None,))[0]
    if res is None or class_names is None or preds_key is None or preds_key not in res:
        return None
    y_true = np.concatenate(res["y_trues_"])
    y_pred = np.concatenate(res[preds_key])
    cm = confusion_matrix(y_true, y_pred, labels=range(len(class_names)))
    with np.errstate(invalid='ignore', divide='ignore'):
        cm_disp = cm / cm.sum(axis=1, keepdims=True)
    return class_names, cm, cm_disp


def nearest_training_target(dist_entry, true_label):
    """Which training-pool target is closest to `true_label`? (None if unavailable)"""
    if dist_entry is None:
        return None
    row_labels, col_labels, mean_mat = dist_entry
    if true_label not in row_labels:
        return None
    row = mean_mat[row_labels.index(true_label)]
    if np.all(np.isnan(row)):
        return None
    return col_labels[int(np.nanargmin(row))]


rows = []
for held_out in CHIPS:
    got = confusion_matrix_for(held_out)
    if got is None:
        print(f"  [!] {short_name(held_out)}: no results for {MODEL}"); continue
    class_names, cm, cm_disp = got
    near_curve = {t: nearest_training_target(curve_dist.get(held_out), t) for t in class_names}
    near_ttp   = {t: nearest_training_target(ct_dist.get(held_out), t) for t in class_names}

    found = False
    for i, t in enumerate(class_names):
        for j, p in enumerate(class_names):
            if i == j or np.isnan(cm_disp[i, j]) or cm_disp[i, j] <= MIN_CONFUSION:
                continue
            found = True
            c_ok = (near_curve[t] == p) if near_curve[t] is not None else None
            t_ok = (near_ttp[t] == p) if near_ttp[t] is not None else None
            rows.append({"chip": short_name(held_out), "true": t, "pred": p,
                         "rate": cm_disp[i, j], "n": int(cm[i, j]),
                         "curve_shape": c_ok, "ttp": t_ok,
                         "either": (bool(c_ok) or bool(t_ok)) if (c_ok is not None or t_ok is not None) else None,
                         "nearest_curve": near_curve[t], "nearest_ttp": near_ttp[t]})
    if not found:
        rows.append({"chip": short_name(held_out), "true": None, "pred": None, "rate": np.nan,
                     "n": 0, "curve_shape": None, "ttp": None, "either": None,
                     "nearest_curve": None, "nearest_ttp": None})

tbl = pd.DataFrame(rows)
real = tbl[tbl["true"].notna()]
print(f"{len(real)} confusions above {MIN_CONFUSION:.0%} for {MODEL}\n")
print(tbl.to_string(index=False))
if len(real):
    print(f"\nCurve shape: {int(real['curve_shape'].sum())}/{len(real)}   "
          f"TTP: {int(real['ttp'].sum())}/{len(real)}   "
          f"Either: {int(real['either'].sum())}/{len(real)}")

In [ ]:
def _yn(v):
    return "N/A" if v is None or (isinstance(v, float) and np.isnan(v)) else ("Yes" if v else "No")

lines = [
    r"\begin{table}[htbp]", r"    \centering",
    r"    \caption{Alignment between concentrated misclassifications (>50\% of a true "
    r"class's held-out pixels) and the nearest training pool class based on curve shape "
    r"and TTP distance. A ``Yes'' confirms that the misclassification direction matches "
    r"the closest training pool target. The ``Either'' column denotes whether curve shape "
    r"or TTP independently explain the confusion.}",
    r"    \label{tab:target_similarity_summary_50}", r"    \small",
    r"    \begin{tabular}{@{}l l c c c@{}}", r"    \toprule",
    r"    \textbf{Chip} & \begin{tabular}[c]{@{}l@{}}\textbf{Dominant Misclassifications} \\"
    r" \textbf{(True $\to$ Predicted)}\end{tabular} & \textbf{Curve Shape} & \textbf{TTP} "
    r"& \textbf{Either} \\",
    r"    \midrule",
]

chips_in_order = sorted(tbl["chip"].unique(), key=chip_sort_key)
for ci, chip in enumerate(chips_in_order):
    g = tbl[tbl["chip"] == chip]
    num = chip.replace("Chip ", "")
    g_real = g[g["true"].notna()]
    if len(g_real) == 0:
        lines.append(f"    {num} & N/A (no confusion $>$50\\%) & N/A & N/A & N/A \\\\")
    else:
        for ri, (_, r) in enumerate(g_real.iterrows()):
            lead = (f"\\multirow{{{len(g_real)}}}{{*}}{{{num}}}" if ri == 0 and len(g_real) > 1
                    else (num if ri == 0 else ""))
            pad = "" if ri == 0 else "                     "
            lines.append(f"    {lead}{pad} & {r['true']} $\\to$ {r['pred']} & "
                         f"{_yn(r['curve_shape'])} & {_yn(r['ttp'])} & {_yn(r['either'])} \\\\")
    if ci < len(chips_in_order) - 1:
        lines.append(r"    \midrule")

n = len(real)
if n:
    lines += [r"    \midrule",
              f"    \\multicolumn{{2}}{{l}}{{\\textbf{{Total (of {n})}}}} & "
              f"\\textbf{{{int(real['curve_shape'].sum())}/{n}}} & "
              f"\\textbf{{{int(real['ttp'].sum())}/{n}}} & "
              f"\\textbf{{{int(real['either'].sum())}/{n}}} \\\\"]
lines += [r"    \bottomrule", r"    \end{tabular}", r"\end{table}"]

latex = "\n".join(lines)
print(latex)

out_path = Path("notebook_rq") / "target_similarity_summary_50.tex"
out_path.write_text(latex + "\n")
print(f"\n[saved] {out_path}")